# Probe & baseline results

Reads `runs/`. No model, no GPU. For looking at results, not producing them.

The question behind all of it: **does the model know enough chess for RL to have
anything to amplify?** RL boosts responses a model already gives sometimes. It
cannot install a missing capability.

In [ ]:
%matplotlib inline
import json
import re
import statistics as st
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt

from plot import INK, INK_SOFT, SURFACE, style

RUNS = Path("runs")
PALETTE = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]
SAN = re.compile(r"^(O-O-O|O-O|[KQRBN]?[a-h]?[1-8]?x?[a-h][1-8](=[QRBN])?[+#]?)$")

# probes: {"0.6b": [...]}  -- newest prefix wins
prefix = "p500" if list(RUNS.glob("p500-*.json")) else "p1024"
probes = {p.stem.split("-", 1)[1]: json.load(open(p))
          for p in sorted(RUNS.glob(f"{prefix}-*.json"))}
order = sorted(probes, key=lambda s: float(s.rstrip("b")))

# baselines: everything that is not a probe
baselines = {p.stem: json.load(open(p)) for p in sorted(RUNS.glob("*.json"))
             if not p.stem.startswith(("probe", "p500", "p1024"))}


def summary(items, kind):
    rows = [i for i in items if i["kind"] == kind]
    return {
        "pct": 100 * sum(i["score"] for i in rows) / len(rows),
        "truncated": sum(i.get("truncated", False) for i in rows),
        "n": len(rows),
        "tokens": st.mean(i.get("tokens", 0) for i in rows),
    }


def model_arm(results):
    return next(a for a in results
                if not a["name"].startswith(("random", "stockfish")))


print(f"probes ({prefix}): {order}")
print(f"baselines: {list(baselines)}")

## 1. Capability probe

`moves` asks where a lone piece can go on an empty board. Does it know the rules?
`board` asks what sits on a square after some movetext. Can it track state?

Half the `board` squares are empty by construction, so **always answering
"empty" scores 50%**. At or below that line is guessing, not tracking.

In [ ]:
for kind in ("moves", "board"):
    print(kind)
    for label in order:
        s = summary(probes[label], kind)
        print(f"  {label:6s} {s['pct']:5.1f}%   truncated {s['truncated']:3d}/{s['n']}"
              f"   mean {s['tokens']:4.0f} tokens")

baseline = 100 * st.mean(
    i["answer"] == "empty" for i in probes[order[0]] if i["kind"] == "board"
)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), facecolor=SURFACE)
for ax, kind, title in zip(
    axes, ("moves", "board"),
    ("Rules: where can this piece go?", "State: what is on this square?"),
):
    pct = [summary(probes[m], kind)["pct"] for m in order]
    bars = ax.bar(order, pct, width=0.55, color=PALETTE[: len(order)], zorder=3)
    for bar, value in zip(bars, pct):
        ax.annotate(f"{value:.0f}%", xy=(bar.get_x() + bar.get_width() / 2, value),
                    xytext=(0, 4), textcoords="offset points",
                    ha="center", color=INK, fontsize=9)
    if kind == "board":
        ax.axhline(baseline, color=INK_SOFT, linestyle=":", linewidth=1, zorder=4)
        ax.annotate(f'always say "empty" ({baseline:.0f}%)',
                    xy=(len(order) - 0.5, baseline), xytext=(0, 5),
                    textcoords="offset points", ha="right",
                    color=INK_SOFT, fontsize=8)
    ax.set_ylim(0, 100)
    ax.set_ylabel("% correct", color=INK_SOFT, fontsize=9)
    ax.set_title(title, color=INK, fontsize=11, loc="left", pad=10)
    style(ax)
    ax.grid(axis="x", visible=False)
fig.tight_layout()

### Two things the score hides

A model can score at the baseline by never attempting, or by attempting and
failing. Those need different fixes. And naming a square outside a1-h8 means it
has movement offsets but no concept of the board's edges.

In [ ]:
OFF_BOARD = re.compile(r"\b[a-h](\d+)\b")

fig, ax = plt.subplots(figsize=(7, 3.0), facecolor=SURFACE)
said_empty, attempted = [], []
for label in order:
    board = [i for i in probes[label] if i["kind"] == "board"]
    n_empty = sum("empty" in i["said"].lower() for i in board)
    said_empty.append(100 * n_empty / len(board))
    attempted.append(100 * (len(board) - n_empty) / len(board))

    moves = [i for i in probes[label] if i["kind"] == "moves"]
    bad = [m.group(0) for i in moves for m in OFF_BOARD.finditer(i["said"].lower())
           if not 1 <= int(m.group(1)) <= 8]
    print(f"{label:6s} off-board squares named: {len(bad):4d}  {sorted(set(bad))[:8]}")

y = range(len(order))
ax.barh(y, said_empty, height=0.5, color=PALETTE[0], zorder=3, label='said "empty"')
ax.barh(y, attempted, height=0.5, left=said_empty, color=PALETTE[1], zorder=3,
        label="named a piece", edgecolor=SURFACE, linewidth=2)
for i, (a, b) in enumerate(zip(said_empty, attempted)):
    for value, left in ((a, 0), (b, a)):
        if value > 8:
            ax.annotate(f"{value:.0f}%", xy=(left + value / 2, i), ha="center",
                        va="center", color="white", fontsize=9)
ax.set_yticks(list(y), order)
ax.set_xlim(0, 100)
ax.set_xlabel("% of board questions", color=INK_SOFT, fontsize=9)
ax.set_title("Does it even attempt the board question?", color=INK,
             fontsize=11, loc="left", pad=10)
ax.legend(loc="lower right", frameon=False, fontsize=9, labelcolor=INK)
style(ax)
ax.grid(axis="y", visible=False)
fig.tight_layout()

### The exact prompt, and what each model said

Verbatim. This is the ground truth behind every number above. Change `index` to
walk through the questions.

In [ ]:
def show(kind, index=0, chars=600):
    ref = [i for i in probes[order[0]] if i["kind"] == kind][index]
    print("=" * 74)
    print(ref["prompt"])
    print(f"\nCORRECT: {ref['answer']}")
    for label in order:
        item = [i for i in probes[label] if i["kind"] == kind][index]
        print("-" * 74)
        print(f"{label}  {'CORRECT' if item['score'] else 'wrong'}   "
              f"{item.get('tokens', 0)} tokens"
              f"{', truncated' if item.get('truncated') else ''}")
        print(f"extracted: {item['said']!r}\n")
        print(item.get("completion", "(not saved)")[:chars], "\n")


show("moves", index=0)

In [ ]:
show("board", index=0, chars=450)

## 2. Baseline runs

Four outcomes. The summary line collapsed them into "answered 95%", which was
mostly the model echoing the prompt's own `MOVE` placeholder.

In [ ]:
OUTCOMES = ["legal move", "illegal move", "boxed non-move", "no answer"]

names, counts = list(baselines), []
for name in names:
    rows = model_arm(baselines[name])["rows"]
    legal = sum(r["legal"] for r in rows)
    illegal = sum(1 for r in rows if r["answered"] and not r["legal"]
                  and SAN.match((r["raw"] or "").strip()))
    junk = sum(1 for r in rows if r["answered"] and not r["legal"]
               and not SAN.match((r["raw"] or "").strip()))
    counts.append([legal, illegal, junk, len(rows) - legal - illegal - junk])
    print(f"{name}: {dict(zip(OUTCOMES, counts[-1]))}")

fig, ax = plt.subplots(figsize=(9, 2.4), facecolor=SURFACE)
left = [0] * len(names)
for i, (outcome, colour) in enumerate(zip(OUTCOMES, PALETTE)):
    values = [c[i] for c in counts]
    ax.barh(names, values, height=0.5, left=left, color=colour, zorder=3,
            label=outcome, edgecolor=SURFACE, linewidth=2)
    for j, value in enumerate(values):
        if value >= 8:
            ax.annotate(str(value), xy=(left[j] + value / 2, j), ha="center",
                        va="center", color="white", fontsize=9)
    left = [l + v for l, v in zip(left, values)]
ax.set_xlabel("positions", color=INK_SOFT, fontsize=9)
ax.set_title("Baseline outcomes", color=INK, fontsize=11, loc="left", pad=10)
ax.legend(loc="center left", bbox_to_anchor=(1.01, 0.5), frameon=False,
          fontsize=9, labelcolor=INK)
style(ax)
ax.grid(axis="y", visible=False)
fig.tight_layout()

## 3. Move quality

Cumulative, so medians read straight off 0.5. Left and up is better. The model
curve sitting right of random means its choices are worse than chance.

In [ ]:
RUN = "movetext-nothink"
COLOURS = {"random": "#eb6834", "stockfish": "#1baf7a", "qwen3": "#2a78d6"}

fig, ax = plt.subplots(figsize=(7, 4), facecolor=SURFACE)
for arm in baselines[RUN]:
    losses = sorted(r["cp_loss"] for r in arm["rows"] if r["cp_loss"] is not None)
    if not losses:
        print(f"{arm['name']}: no legal moves")
        continue
    key = ("random" if arm["name"].startswith("random")
           else "stockfish" if arm["name"].startswith("stockfish") else "qwen3")
    good = sum(x <= 50 for x in losses) / len(losses)
    print(f"{arm['name'][:34]:36s} n={len(losses):3d}  mean {st.mean(losses):5.0f}"
          f"  median {st.median(losses):5.0f}  good {good:5.1%}")

    fraction = [(i + 1) / len(losses) for i in range(len(losses))]
    ax.step(losses, fraction, where="post", color=COLOURS[key], linewidth=2,
            label=f"{key} (n={len(losses)})")
    at = next((i for i, f in enumerate(fraction) if f >= 0.6), len(losses) - 1)
    ax.annotate(key, xy=(losses[at], fraction[at]), xytext=(8, -4),
                textcoords="offset points", color=INK, fontsize=9)

ax.axvline(50, color=INK_SOFT, linestyle=":", linewidth=1)
ax.set_xlabel("centipawns lost vs Stockfish's best", color=INK_SOFT, fontsize=9)
ax.set_ylabel("fraction of positions", color=INK_SOFT, fontsize=9)
ax.set_title(f"Move quality: {RUN}", color=INK, fontsize=11, loc="left", pad=10)
ax.set_ylim(0, 1.02)
ax.legend(loc="lower right", frameon=False, fontsize=9, labelcolor=INK)
style(ax)
fig.tight_layout()

## 4. Degenerate-policy check

A published GRPO run on 8B models converged on pushing the a-pawn over 80% of
the time. Nearly always legal, rarely catastrophic, and the best constant answer
when you can't read a board. Mean cp_loss improves the whole way. **This is what
separates learning from collapse.**

In [ ]:
played = [r["move"] for r in model_arm(baselines[RUN])["rows"] if r.get("move")]

if not played:
    print(f"{RUN}: no legal moves to plot")
else:
    counts = Counter(played).most_common(12)
    labels = [m for m, _ in counts][::-1]
    shares = [100 * n / len(played) for _, n in counts][::-1]
    top, n_top = counts[0]
    share = 100 * n_top / len(played)

    fig, ax = plt.subplots(figsize=(7, 4), facecolor=SURFACE)
    ax.barh(labels, shares, height=0.6, color=PALETTE[0], zorder=3)
    for y, value in enumerate(shares):
        ax.annotate(f"{value:.0f}%", xy=(value, y), xytext=(4, 0),
                    textcoords="offset points", va="center", color=INK, fontsize=8)
    ax.set_title(f"Moves played ({len(played)} legal). top: {top} at {share:.0f}%"
                 f"{'  <- degenerate' if share >= 20 else ''}",
                 color=INK, fontsize=11, loc="left", pad=10)
    ax.set_xlabel("% of legal answers", color=INK_SOFT, fontsize=9)
    style(ax)
    ax.grid(axis="y", visible=False)
    fig.tight_layout()

## 5. Read a completion

Numbers say a run failed. Only the text says why.

In [ ]:
row = model_arm(baselines[RUN])["rows"][0]
print(f"raw    {row['raw']!r}")
print(f"move   {row['move']}   legal={row['legal']}  cp_loss={row['cp_loss']}")
print(f"tokens {row['tokens']}  truncated={row['truncated']}")
print("\n--- completion ---")
print(row["completion"][:1200])